# Contact Map Analysis — CESA3 / HvGlu3 Complex

This notebook computes:
1. **Overall** contact frequency map (all 1644 frames, 4 trajectories)
2. **Per-cluster** contact maps (K=4 clusters from K-means analysis)
3. **Residue × Cluster heatmap** — which HvGlu3 residues are active in which cluster
4. **CESA1 comparison** — are LEU74/VAL77 still the anchor? Does Cluster 3 recruit the deep contacts?

### Cluster key (from K-means PCA_clusters.csv)
| CSV label | Bar chart label | Frames | % | Identity |
|---|---|---|---|---|
| 0 | Cluster 1 | 307 | 18.7% | Distal sub-state |
| 1 | Cluster 2 | 525 | 31.9% | Distal sub-state |
| **2** | **Cluster 3** | **370** | **22.5%** | **Close-binding state (T1, PC1=-355)** |
| 3 | Cluster 4 | 442 | 26.9% | Intermediate/transitional |

### Chain layout
- **Chain 0 (A)**: HvGlu3, 619 residues, resSeq 1–619
- **Chain 1 (B)**: CESA3 subunit 1, 808 residues, resSeq 254–1061
- **Chain 2 (C)**: CESA3 subunit 2, 808 residues, resSeq 1062–1869
- **Chain 3 (D)**: CESA3 subunit 3, 808 residues, resSeq 1870–2677

In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
%matplotlib inline

import mdtraj as md
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family']     = 'sans-serif'

print('mdtraj version:', md.__version__)
print('All imports OK!')

In [ ]:
# ============================================================
# CELL 2 — File paths and parameters
# ============================================================

BASE_DIR   = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')
TOPOLOGY   = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/run/protein.pdb')
TRAJECTORY = BASE_DIR / 'CESA3_all_protein.xtc'
KMEANS_CSV = BASE_DIR / 'Kmeans' / 'PCA_clusters.csv'
OUT_DIR    = BASE_DIR / 'ContactMap'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis parameters
CUTOFF_NM = 0.45   # 4.5 Å — same as CESA1 analysis
STRIDE    = 1      # use every frame (1644 total — manageable with chunked mdtraj)
CHUNK     = 50     # HvGlu3 residues per batch (memory efficiency)

# Cluster labels for display (CSV uses 0-indexed, display uses 1-indexed)
CLUSTER_LABELS = {0: 'Cluster 1 (18.7%)', 1: 'Cluster 2 (31.9%)',
                  2: 'Cluster 3 (22.5%) — close-binding', 3: 'Cluster 4 (26.9%)'}
CLOSE_CLUSTER  = 2   # CSV index of the close-binding state (T1 cluster)

# Verify files
for f in [TOPOLOGY, TRAJECTORY, KMEANS_CSV]:
    status = '✓' if f.exists() else '✗ MISSING'
    size   = f'{f.stat().st_size / 1e6:.1f} MB' if f.exists() else ''
    print(f'  {status}  {f.name}  {size}')

print(f'\nCutoff       : {CUTOFF_NM*10:.1f} Å  ({CUTOFF_NM} nm)')
print(f'Stride       : {STRIDE}')
print(f'Output folder: {OUT_DIR}')

In [ ]:
# ============================================================
# CELL 3 — Load topology only (no full trajectory in RAM)
# Chain info is derived from the PDB topology file.
# The trajectory will be streamed in chunks in Cell 6.
# ============================================================

print('Loading topology from PDB (no trajectory needed yet)...')
top = md.load_topology(str(TOPOLOGY))
print('Topology loaded.')

print('\nChain structure:')
chain_info = {}
for ci, chain in enumerate(top.chains):
    res_list = list(chain.residues)
    chain_info[ci] = {
        'residues': [r.index for r in res_list],
        'resSeqs':  [r.resSeq for r in res_list],
        'resnames': [r.name  for r in res_list],
        'n':        len(res_list),
        'first':    res_list[0].resSeq,
        'last':     res_list[-1].resSeq
    }
    label = ['HvGlu3','CESA3-B','CESA3-C','CESA3-D'][ci]
    print(f'  Chain {ci} ({label}): {len(res_list)} residues, '
          f'resSeq {res_list[0].resSeq}-{res_list[-1].resSeq}')

# Residue indices for contact computation
hvglu3_idx  = np.array(chain_info[0]['residues'])
hvglu3_seq  = np.array(chain_info[0]['resSeqs'])
hvglu3_name = np.array(chain_info[0]['resnames'])

cesa3_idx   = np.concatenate([chain_info[1]['residues'],
                               chain_info[2]['residues'],
                               chain_info[3]['residues']])
cesa3_seq   = np.concatenate([chain_info[1]['resSeqs'],
                               chain_info[2]['resSeqs'],
                               chain_info[3]['resSeqs']])
cesa3_name  = np.concatenate([chain_info[1]['resnames'],
                               chain_info[2]['resnames'],
                               chain_info[3]['resnames']])
cesa3_chain = np.concatenate([['B']*chain_info[1]['n'],
                               ['C']*chain_info[2]['n'],
                               ['D']*chain_info[3]['n']])

print(f'\nHvGlu3 residues : {len(hvglu3_idx)}')
print(f'CESA3 residues  : {len(cesa3_idx)} (B+C+D)')
print(f'Total pairs     : {len(hvglu3_idx)*len(cesa3_idx):,}')
print('\nTrajectory NOT loaded into RAM -- will stream via md.iterload() in Cell 6')


In [ ]:
# ============================================================
# CELL 4 — Load cluster assignments
# ============================================================

df_clusters = pd.read_csv(KMEANS_CSV)
print(f'Cluster assignments: {len(df_clusters)} frames')
print(df_clusters['Cluster'].value_counts().sort_index())

assert len(df_clusters) == len(traj), \
    f'Frame mismatch: CSV={len(df_clusters)}, traj={len(traj)}'
print('\n✓ Frame count matches trajectory')

cluster_frame_idx = {}
for c in sorted(df_clusters['Cluster'].unique()):
    cluster_frame_idx[c] = df_clusters[df_clusters['Cluster'] == c].index.tolist()
    print(f'  Cluster {c+1} (CSV={c}): {len(cluster_frame_idx[c])} frames')

In [ ]:
# ============================================================
# CELL 5 — Core contact frequency function (memory-efficient)
#
# Processes HvGlu3 residues in chunks of CHUNK to avoid
# loading the full distance matrix into memory at once.
# Returns: contact_freq matrix (n_hvglu3, n_cesa3)
# ============================================================

def compute_contact_freq(traj_sub, hvglu3_idx, cesa3_idx, cutoff_nm=0.45, chunk=50):
    n_hvglu3 = len(hvglu3_idx)
    n_cesa3  = len(cesa3_idx)
    n_frames = len(traj_sub)
    contact_count = np.zeros((n_hvglu3, n_cesa3), dtype=np.int32)

    for start in range(0, n_hvglu3, chunk):
        end   = min(start + chunk, n_hvglu3)
        chunk_hv = hvglu3_idx[start:end]
        pairs    = np.array([(int(i), int(j)) for i in chunk_hv for j in cesa3_idx])
        dist, _  = md.compute_contacts(traj_sub, contacts=pairs,
                                        scheme='closest-heavy', periodic=True)
        # dist shape: (n_frames, n_pairs_in_chunk)
        in_contact = (dist < cutoff_nm)  # bool
        counts = in_contact.sum(axis=0).reshape(end - start, n_cesa3)
        contact_count[start:end, :] = counts
        pct = end / n_hvglu3 * 100
        print(f'  HvGlu3 res {start:3d}–{end-1:3d} / {n_hvglu3}  ({pct:.0f}%)', flush=True)

    return contact_count / n_frames

print('Contact frequency function defined!')
print(f'Cutoff: {CUTOFF_NM*10} Å  |  Chunk size: {CHUNK} HvGlu3 residues per batch')

In [ ]:
# ============================================================
# CELL 6 — Compute OVERALL + all 4 cluster contact maps in ONE
# pass using md.iterload() -- avoids loading full trajectory.
#
# Peak RAM per iteration:
#   traj chunk  : TRAJ_CHUNK * 48455 atoms * 3 * 4 B  ~ 58 MB (at 100 frames)
#   dist array  : TRAJ_CHUNK * CHUNK * n_cesa3 * 4 B  ~ 29 MB
#   Total peak  : < 120 MB  (vs ~1.75 GB with md.load)
# ============================================================

TRAJ_CHUNK = 100   # frames per iterload iteration

npy_overall = OUT_DIR / 'contact_freq_overall.npy'
npy_cluster = {c: OUT_DIR / f'contact_freq_cluster{c}.npy' for c in range(4)}
all_exist   = npy_overall.exists() and all(p.exists() for p in npy_cluster.values())

if all_exist:
    print('Loading pre-computed maps from .npy files...')
    cf_overall = np.load(npy_overall)
    cf_cluster = {c: np.load(npy_cluster[c]) for c in range(4)}
    print('Done.')
else:
    cluster_arr   = df_clusters['Cluster'].values   # shape (1644,)
    n_hv          = len(hvglu3_idx)
    n_ce          = len(cesa3_idx)
    count_overall = np.zeros((n_hv, n_ce), dtype=np.int32)
    count_cluster = {c: np.zeros((n_hv, n_ce), dtype=np.int32) for c in range(4)}
    n_cluster     = {c: 0 for c in range(4)}
    total_frames  = 0
    frame_offset  = 0

    print(f'Streaming trajectory in chunks of {TRAJ_CHUNK} frames...')
    for chunk_traj in md.iterload(str(TRAJECTORY), top=str(TOPOLOGY), chunk=TRAJ_CHUNK):
        n_chunk = len(chunk_traj)
        labels  = cluster_arr[frame_offset:frame_offset + n_chunk]

        for start in range(0, n_hv, CHUNK):
            end      = min(start + CHUNK, n_hv)
            chunk_hv = hvglu3_idx[start:end]
            pairs    = np.array([(int(i), int(j)) for i in chunk_hv for j in cesa3_idx])
            dist, _  = md.compute_contacts(chunk_traj, contacts=pairs,
                                            scheme='closest-heavy', periodic=True)
            in_contact = (dist < CUTOFF_NM)   # (n_chunk, (end-start)*n_ce)

            # accumulate overall
            count_overall[start:end, :] += in_contact.sum(axis=0).reshape(end - start, n_ce)

            # accumulate per-cluster
            for c in range(4):
                m = (labels == c)
                if m.any():
                    count_cluster[c][start:end, :] += (
                        in_contact[m].sum(axis=0).reshape(end - start, n_ce))

        for c in range(4):
            n_cluster[c] += int((labels == c).sum())
        total_frames += n_chunk
        frame_offset += n_chunk
        pct = frame_offset / 1644 * 100
        print(f'  Frames {frame_offset:4d}/1644  ({pct:.0f}%)', flush=True)

    cf_overall = count_overall / total_frames
    cf_cluster = {c: count_cluster[c] / max(n_cluster[c], 1) for c in range(4)}

    np.save(npy_overall, cf_overall)
    for c in range(4):
        np.save(npy_cluster[c], cf_cluster[c])
    print(f'\nSaved overall + 4 cluster .npy files')

    # rebuild cluster_frame_idx for downstream cells
    cluster_frame_idx = {}
    for c in range(4):
        cluster_frame_idx[c] = list(np.where(df_clusters['Cluster'].values == c)[0])

print(f'\nMatrix shape : {cf_overall.shape}  (HvGlu3 x CESA3)')
print(f'Contacts >30%: {(cf_overall > 0.3).sum():,}')
print(f'Contacts >50%: {(cf_overall > 0.5).sum():,}')
print(f'Contacts >70%: {(cf_overall > 0.7).sum():,}')
for c in range(4):
    n30 = (cf_cluster[c] > 0.3).sum()
    n50 = (cf_cluster[c] > 0.5).sum()
    print(f'  Cluster {c+1}: >30%={n30:,}  >50%={n50:,}')


In [ ]:
# ============================================================
# CELL 7 — Per-cluster maps are computed in Cell 6 (iterload pass).
# This cell rebuilds cluster_frame_idx if the maps were loaded
# from .npy cache (i.e., Cell 6 took the fast path).
# ============================================================

if 'cluster_frame_idx' not in dir():
    cluster_frame_idx = {}
    for c in range(4):
        cluster_frame_idx[c] = list(np.where(df_clusters['Cluster'].values == c)[0])

print('All cluster contact maps ready:')
for c in sorted(cf_cluster.keys()):
    n30 = (cf_cluster[c] > 0.3).sum()
    n50 = (cf_cluster[c] > 0.5).sum()
    print(f'  Cluster {c+1}: contacts>30%={n30:,}  contacts>50%={n50:,}')


In [ ]:
# ============================================================
# CELL 8 — Extract top contacts (threshold-filtered pairs)
# ============================================================

def extract_contacts(cf_matrix, hvglu3_seq, hvglu3_name,
                     cesa3_seq, cesa3_name, cesa3_chain,
                     threshold=0.30):
    rows = []
    idxs = np.argwhere(cf_matrix > threshold)
    for (i, j) in idxs:
        rows.append({
            'HVGlu3_resid':   int(hvglu3_seq[i]),
            'HVGlu3_resname': hvglu3_name[i],
            'CESA3_chain':    cesa3_chain[j],
            'CESA3_resid':    int(cesa3_seq[j]),
            'CESA3_resname':  cesa3_name[j],
            'frequency':      round(float(cf_matrix[i, j]), 3)
        })
    return pd.DataFrame(rows).sort_values('frequency', ascending=False).reset_index(drop=True)

# Overall contacts
df_overall = extract_contacts(cf_overall, hvglu3_seq, hvglu3_name,
                               cesa3_seq, cesa3_name, cesa3_chain, threshold=0.50)
df_overall.to_csv(OUT_DIR / 'contacts_overall_freq50.csv', index=False)

# Per-cluster contacts
df_clust = {}
for c in sorted(cf_cluster.keys()):
    df_clust[c] = extract_contacts(cf_cluster[c], hvglu3_seq, hvglu3_name,
                                    cesa3_seq, cesa3_name, cesa3_chain, threshold=0.30)
    df_clust[c].to_csv(OUT_DIR / f'contacts_cluster{c+1}_freq30.csv', index=False)

print(f'Overall contacts (>50%): {len(df_overall)}')
for c in sorted(df_clust.keys()):
    print(f'Cluster {c+1} contacts (>30%): {len(df_clust[c])}')

print('\nTop 15 overall contacts:')
print(df_overall.head(15).to_string(index=False))

In [ ]:
# ============================================================
# CELL 9 — Dot matrix plot function (matches CESA1 style)
# ============================================================

def plot_dot_contact_map(cf_matrix, hvglu3_seq, cesa3_seq, cesa3_chain,
                          title, output_path, threshold=0.50):
    """
    Dot matrix where each dot = one residue pair with freq > threshold.
    Dot size and colour encode contact frequency.
    Y-axis split by CESA3 chain (B / C / D).
    """
    # Identify contacts above threshold
    mask = cf_matrix > threshold
    hi_idx = np.argwhere(mask)
    if len(hi_idx) == 0:
        print(f'No contacts above {threshold:.0%} — skipping {title}')
        return

    freqs  = cf_matrix[mask]
    hv_pos = hvglu3_seq[hi_idx[:, 0]]
    ce_pos = cesa3_seq[hi_idx[:, 1]]
    chains = cesa3_chain[hi_idx[:, 1]]

    # Build y-positions: continuous axis with chain offsets
    chain_order = ['B', 'C', 'D']
    chain_color = {'B': '#2166ac', 'C': '#d6604d', 'D': '#4dac26'}
    n_B = (cesa3_chain == 'B').sum()
    n_C = (cesa3_chain == 'C').sum()
    n_D = (cesa3_chain == 'D').sum()

    offset = {'B': 0, 'C': n_B, 'D': n_B + n_C}

    # Convert residue seq to position index within chain
    seq_to_pos = {}
    for chain in chain_order:
        mask_c = cesa3_chain == chain
        for local_i, seq in enumerate(cesa3_seq[mask_c]):
            seq_to_pos[(chain, int(seq))] = local_i + offset[chain]

    y_positions = np.array([seq_to_pos.get((ch, int(s)), np.nan)
                             for ch, s in zip(chains, ce_pos)])
    valid = ~np.isnan(y_positions)

    fig, ax = plt.subplots(figsize=(14, 9))

    sc = ax.scatter(
        hv_pos[valid], y_positions[valid],
        c=freqs[valid], cmap='YlOrRd',
        s=freqs[valid]**2 * 300,
        alpha=0.85, vmin=threshold, vmax=1.0,
        linewidths=0.3, edgecolors='grey'
    )
    plt.colorbar(sc, ax=ax, label='Contact frequency', fraction=0.02, pad=0.02)

    # Chain boundary lines
    boundaries   = [n_B - 0.5, n_B + n_C - 0.5]
    tick_centers = [n_B / 2, n_B + n_C / 2, n_B + n_C + n_D / 2]
    tick_labels  = ['CESA3 subunit B', 'CESA3 subunit C', 'CESA3 subunit D']
    for b in boundaries:
        ax.axhline(b, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.set_yticks(tick_centers)
    ax.set_yticklabels(tick_labels, fontsize=10)

    ax.set_xlabel('HvGlu3 residue (resSeq)', fontsize=12, labelpad=8)
    ax.set_title(title, fontsize=13, weight='bold', pad=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {Path(output_path).name}')

print('Dot matrix plot function defined!')

In [ ]:
# ============================================================
# CELL 10 — Plot overall contact map
# ============================================================

plot_dot_contact_map(
    cf_overall, hvglu3_seq, cesa3_seq, cesa3_chain,
    title   = 'CESA3 — HvGlu3 Overall Contact Map (all frames, ≥50%)',
    output_path = OUT_DIR / 'contact_map_overall.png',
    threshold   = 0.50
)

In [ ]:
# ============================================================
# CELL 11 — Plot per-cluster contact maps
# ============================================================

for c in sorted(cf_cluster.keys()):
    label = CLUSTER_LABELS[c]
    plot_dot_contact_map(
        cf_cluster[c], hvglu3_seq, cesa3_seq, cesa3_chain,
        title       = f'CESA3 — HvGlu3 Contact Map  {label}  (≥30%)',
        output_path = OUT_DIR / f'contact_map_cluster{c+1}.png',
        threshold   = 0.30
    )

In [ ]:
# ============================================================
# CELL 12 — Residue × Cluster heatmap
#
# Rows = HvGlu3 residues active in ≥1 cluster
# Cols = 4 conformational clusters
# Value = max contact frequency with any CESA3 residue in that cluster
# ============================================================

# Collect all HvGlu3 residues that make any contact > 30% in any cluster
active_mask = np.zeros(len(hvglu3_idx), dtype=bool)
for c in sorted(cf_cluster.keys()):
    active_mask |= (cf_cluster[c].max(axis=1) > 0.30)

active_hv_positions = np.where(active_mask)[0]
active_hv_seq       = hvglu3_seq[active_mask]
active_hv_name      = hvglu3_name[active_mask]

print(f'HvGlu3 residues active in ≥1 cluster (>30%): {active_mask.sum()}')

# Build heatmap matrix
n_active = active_mask.sum()
n_clust  = len(cf_cluster)
heatmap  = np.zeros((n_active, n_clust))
n_contacts_per_clust = np.zeros((n_active, n_clust), dtype=int)

for ci, c in enumerate(sorted(cf_cluster.keys())):
    sub = cf_cluster[c][active_mask, :]   # (n_active, n_cesa3)
    heatmap[:, ci] = sub.max(axis=1)      # max freq with any CESA3 residue
    n_contacts_per_clust[:, ci] = (sub > 0.30).sum(axis=1)

# Row labels: resid + resname
row_labels = [f'{name}{seq}' for name, seq in zip(active_hv_name, active_hv_seq)]

# Column labels
col_labels = [f'Cluster {c+1}\n({["18.7%","31.9%","22.5%","26.9%"][c]})'
               for c in sorted(cf_cluster.keys())]

# Plot
fig, ax = plt.subplots(figsize=(8, max(10, n_active * 0.32)))

im = ax.imshow(heatmap, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1,
                interpolation='nearest')
plt.colorbar(im, ax=ax, label='Max contact frequency', fraction=0.02, pad=0.03)

ax.set_xticks(range(n_clust))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(range(n_active))
ax.set_yticklabels(row_labels, fontsize=8)
ax.set_title('CESA3 — HvGlu3 Residue × Cluster Contact Heatmap', fontsize=12, weight='bold')

# Annotate cells with frequency value
for i in range(n_active):
    for j in range(n_clust):
        val = heatmap[i, j]
        if val > 0.05:
            color = 'white' if val > 0.7 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=color)

# Highlight the close-binding cluster column
close_col = list(sorted(cf_cluster.keys())).index(CLOSE_CLUSTER)
ax.add_patch(plt.Rectangle((close_col - 0.5, -0.5), 1, n_active,
                             fill=False, edgecolor='navy', lw=2.5))

# Mark anchor residues from CESA1 (LEU74, VAL77)
anchor_seqs = {74, 77}
for ri, seq in enumerate(active_hv_seq):
    if seq in anchor_seqs:
        ax.annotate('◀ anchor', (n_clust - 0.45, ri), fontsize=7,
                    color='darkred', va='center')

plt.tight_layout()
out = OUT_DIR / 'Fig_residue_cluster_heatmap.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# ============================================================
# CELL 13 — Combined 4-panel figure (A=overall, B=pop bar, C=clusters)
# Matches CESA1 publication figure style
# ============================================================

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 22))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.3)

cluster_pops = {0: 18.7, 1: 31.9, 2: 22.5, 3: 26.9}
cluster_colors = ['#4393c3', '#2ca25f', '#8b4513', '#888888']

# Panel A: overall contact map (dot matrix) — text reference
ax_A = fig.add_subplot(gs[0, :])
threshold_A = 0.50
mask_A  = cf_overall > threshold_A
hi_idx  = np.argwhere(mask_A)
freqs_A = cf_overall[mask_A]
hv_A    = hvglu3_seq[hi_idx[:, 0]]

n_B = (cesa3_chain == 'B').sum()
n_C = (cesa3_chain == 'C').sum()
n_D = (cesa3_chain == 'D').sum()
offset = {'B': 0, 'C': n_B, 'D': n_B + n_C}
seq_to_pos = {}
for chain in ['B', 'C', 'D']:
    m = cesa3_chain == chain
    for li, sq in enumerate(cesa3_seq[m]):
        seq_to_pos[(chain, int(sq))] = li + offset[chain]

ce_ch_A = cesa3_chain[hi_idx[:, 1]]
ce_sq_A = cesa3_seq[hi_idx[:, 1]]
y_A = np.array([seq_to_pos.get((ch, int(s)), np.nan) for ch, s in zip(ce_ch_A, ce_sq_A)])
valid = ~np.isnan(y_A)

sc = ax_A.scatter(hv_A[valid], y_A[valid], c=freqs_A[valid], cmap='YlOrRd',
                   s=freqs_A[valid]**2 * 250, alpha=0.85,
                   vmin=threshold_A, vmax=1.0, linewidths=0.3, edgecolors='grey')
plt.colorbar(sc, ax=ax_A, label='Contact frequency', fraction=0.015, pad=0.02)
for b in [n_B - 0.5, n_B + n_C - 0.5]:
    ax_A.axhline(b, color='black', lw=0.8, ls='--', alpha=0.5)
ax_A.set_yticks([n_B/2, n_B+n_C/2, n_B+n_C+n_D/2])
ax_A.set_yticklabels(['CESA3 subunit B', 'CESA3 subunit C', 'CESA3 subunit D'], fontsize=9)
ax_A.set_xlabel('HvGlu3 residue', fontsize=11)
ax_A.set_title('(A) Overall contact map — all frames ≥50%', fontsize=11, weight='bold')
ax_A.spines['top'].set_visible(False); ax_A.spines['right'].set_visible(False)

# Panel B: cluster populations
ax_B = fig.add_subplot(gs[1, 0])
bars = ax_B.bar(range(1, 5), [cluster_pops[c] for c in range(4)],
                 color=cluster_colors, width=0.6, edgecolor='black', lw=0.5)
for bar, val in zip(bars, cluster_pops.values()):
    ax_B.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
              f'{val}%', ha='center', va='bottom', fontsize=10, weight='bold')
ax_B.set_xlabel('Conformational cluster', fontsize=11)
ax_B.set_ylabel('Population (%)', fontsize=11)
ax_B.set_title('(B) Cluster populations (K=4)', fontsize=11, weight='bold')
ax_B.set_xticks(range(1, 5))
ax_B.spines['top'].set_visible(False); ax_B.spines['right'].set_visible(False)

# Panel C: heatmap
ax_C = fig.add_subplot(gs[1, 1])
im = ax_C.imshow(heatmap, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax_C, label='Max contact freq', fraction=0.03)
ax_C.set_xticks(range(n_clust))
ax_C.set_xticklabels([f'C{c+1}' for c in sorted(cf_cluster.keys())], fontsize=9)
ax_C.set_yticks(range(n_active))
ax_C.set_yticklabels(row_labels, fontsize=7)
ax_C.set_title('(C) Residue × Cluster heatmap', fontsize=11, weight='bold')
close_col_idx = list(sorted(cf_cluster.keys())).index(CLOSE_CLUSTER)
ax_C.add_patch(plt.Rectangle((close_col_idx - 0.5, -0.5), 1, n_active,
                               fill=False, edgecolor='navy', lw=2.0))
for i in range(n_active):
    for j in range(n_clust):
        v = heatmap[i, j]
        if v > 0.05:
            ax_C.text(j, i, f'{v:.2f}', ha='center', va='center',
                      fontsize=6, color='white' if v > 0.7 else 'black')

plt.suptitle('CESA3 — HvGlu3 Contact Map Analysis', fontsize=14, weight='bold', y=1.01)
out_combined = OUT_DIR / 'Fig_combined_overall_and_clusters.png'
fig.savefig(out_combined, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_combined.name}')

In [ ]:
# ============================================================
# CELL 14 — CESA1 comparison: answer the 3 key questions
# ============================================================

print('=' * 70)
print('COMPARISON WITH CESA1 — answering the 3 key questions')
print('=' * 70)

# ---- Q1: Are LEU74 and VAL77 the absolute anchor? ----
print('\n--- Q1: Are LEU74 and VAL77 the universal anchor in CESA3? ---')
anchor_cesa1 = [74, 77]   # both had freq=1.00 in all 4 CESA1 clusters

for seq in anchor_cesa1:
    pos = np.where(hvglu3_seq == seq)[0]
    if len(pos) == 0:
        print(f'  Residue {seq}: NOT FOUND in HvGlu3')
        continue
    i = pos[0]
    name = hvglu3_name[i]
    print(f'  {name}{seq}:')
    print(f'    Overall max freq : {cf_overall[i].max():.3f} (vs CESA1: 1.000)')
    for c in sorted(cf_cluster.keys()):
        max_f = cf_cluster[c][i].max()
        marker = ' ← CLOSE STATE' if c == CLOSE_CLUSTER else ''
        print(f'    Cluster {c+1}        : {max_f:.3f}{marker}')

# ---- Q2: Does Cluster 3 (close-binding) recruit the "deep" contacts? ----
print('\n--- Q2: Does Cluster 3 recruit the deep contacts (PHE81, ILE84, PRO609-613)? ---')
deep_residues_cesa1 = [81, 84, 88, 609, 610, 611, 612, 613]

for seq in deep_residues_cesa1:
    pos = np.where(hvglu3_seq == seq)[0]
    if len(pos) == 0:
        print(f'  Residue {seq}: NOT FOUND')
        continue
    i = pos[0]
    name = hvglu3_name[i]
    freqs_by_clust = {c: cf_cluster[c][i].max() for c in sorted(cf_cluster.keys())}
    close_f = freqs_by_clust[CLOSE_CLUSTER]
    other_f = np.mean([v for k, v in freqs_by_clust.items() if k != CLOSE_CLUSTER])
    delta   = close_f - other_f
    tag = '  ★ UNIQUE TO CLOSE STATE' if close_f > 0.5 and other_f < 0.3 else \
          '  ↑ enriched in close state' if delta > 0.2 else \
          '  ↔ similar across clusters' if abs(delta) < 0.1 else ''
    vals = '  '.join([f'C{c+1}:{freqs_by_clust[c]:.2f}' for c in sorted(freqs_by_clust)])
    print(f'  {name}{seq:>4d}:  {vals}{tag}')

# ---- Q3: What does the distal state contact map look like? ----
print('\n--- Q3: Core interface in distal states (Clusters 1, 2, 4) ---')
print('Residues with max_freq > 0.70 in the distal clusters (0, 1, 3):')
distal_clusters = [c for c in sorted(cf_cluster.keys()) if c != CLOSE_CLUSTER]

rows_q3 = []
for i, seq in enumerate(hvglu3_seq):
    max_distal = max(cf_cluster[c][i].max() for c in distal_clusters)
    max_close  = cf_cluster[CLOSE_CLUSTER][i].max()
    if max_distal > 0.70:
        rows_q3.append({
            'residue':     f'{hvglu3_name[i]}{seq}',
            'max_distal':  round(max_distal, 3),
            'max_close':   round(max_close, 3),
            'delta(close-distal)': round(max_close - max_distal, 3)
        })

df_q3 = pd.DataFrame(rows_q3).sort_values('max_distal', ascending=False)
print(df_q3.to_string(index=False))

In [ ]:
# ============================================================
# CELL 15 — Key contacts: close vs distal (CSV export)
# ============================================================

# HvGlu3 key residues summary
rows_key = []
for i, (seq, name) in enumerate(zip(hvglu3_seq, hvglu3_name)):
    overall_f  = cf_overall[i].max()
    close_f    = cf_cluster[CLOSE_CLUSTER][i].max()
    distal_f   = max(cf_cluster[c][i].max() for c in range(4) if c != CLOSE_CLUSTER)
    if max(overall_f, close_f, distal_f) > 0.20:
        rows_key.append({
            'HVGlu3_resid':  int(seq),
            'HVGlu3_resname': name,
            'freq_overall':   round(overall_f, 3),
            'freq_close_C3':  round(close_f,  3),
            'freq_distal_max': round(distal_f, 3),
            'delta':          round(close_f - distal_f, 3)
        })

df_key = pd.DataFrame(rows_key).sort_values('freq_overall', ascending=False)
df_key.to_csv(OUT_DIR / 'hvglu3_key_residues_CESA3.csv', index=False)

print('HvGlu3 key residues (overall max contact freq > 20%):')
print(df_key.to_string(index=False))

# Close-state specific contacts
df_close = df_clust[CLOSE_CLUSTER].head(30)
df_close.to_csv(OUT_DIR / 'key_contacts_cluster3_close.csv', index=False)
print(f'\nTop contacts in Cluster 3 (close-binding state):')
print(df_close.head(20).to_string(index=False))

In [ ]:
# ============================================================
# CELL 16 — Headline numbers for paper
# ============================================================

print('=' * 65)
print('HEADLINE NUMBERS — CESA3 / HvGlu3 contact analysis')
print('=' * 65)

# Total contacts
n_overall_30 = (cf_overall > 0.30).any(axis=1).sum()
n_overall_50 = (cf_overall > 0.50).any(axis=1).sum()
n_close_30   = (cf_cluster[CLOSE_CLUSTER] > 0.30).any(axis=1).sum()

print(f'HvGlu3 residues making any contact > 30% (overall): {n_overall_30}')
print(f'HvGlu3 residues making any contact > 50% (overall): {n_overall_50}')
print(f'HvGlu3 residues in close-binding state (C3, >30%):  {n_close_30}')

# Anchor residues
print('\nAnchor check (LEU74, VAL77):')
for seq in [74, 77]:
    pos = np.where(hvglu3_seq == seq)[0]
    if len(pos):
        i = pos[0]
        print(f'  {hvglu3_name[i]}{seq}: overall={cf_overall[i].max():.3f}  '
              f'close={cf_cluster[CLOSE_CLUSTER][i].max():.3f}')

# CESA3 chains involved
df50 = extract_contacts(cf_overall, hvglu3_seq, hvglu3_name,
                         cesa3_seq, cesa3_name, cesa3_chain, threshold=0.50)
print('\nCESA3 chains contributing to contacts (>50%):')
print(df50.groupby('CESA3_chain').size().to_string())

# Unique contacts in close vs distal
close_active  = set(zip(df_clust[CLOSE_CLUSTER]['HVGlu3_resid'],
                         df_clust[CLOSE_CLUSTER]['CESA3_resid']))
distal_active = set()
for c in range(4):
    if c != CLOSE_CLUSTER:
        distal_active |= set(zip(df_clust[c]['HVGlu3_resid'], df_clust[c]['CESA3_resid']))

close_only  = close_active  - distal_active
distal_only = distal_active - close_active
shared      = close_active  & distal_active

print(f'\nContact pairs unique to close state (C3 only): {len(close_only)}')
print(f'Contact pairs unique to distal states only:   {len(distal_only)}')
print(f'Contact pairs shared between all states:      {len(shared)}')

---
## Representative Structure Analysis - 4 Cluster PDBs

Each cluster centroid is extracted from the XTC trajectory using the frame index in the filename.
(The PDB files written by mdtraj contain a non-UTF8 byte that blocks re-loading;
extracting the same frame from the XTC gives identical coordinates.)

| PDB file | XTC frame | Cluster |
|---|---|---|
| cluster_1_frame_723.pdb  | 723  | Cluster 1 - distal |
| cluster_2_frame_1328.pdb | 1328 | Cluster 2 - distal |
| cluster_3_frame_199.pdb  | 199  | Cluster 3 - close-binding (T1) |
| cluster_4_frame_1058.pdb | 1058 | Cluster 4 - intermediate |


In [ ]:
# CELL 17 - Load 4 representative frames using md.load_frame()
# (loads only the specific frame index, not the full trajectory)
import re
PDB_DIR = BASE_DIR / 'Kmeans' / 'pdbs'
rep_frames = {}
for fname in sorted(PDB_DIR.glob('cluster_*.pdb')):
    m = re.search(r'cluster_(\d+)_frame_(\d+)\.pdb', fname.name)
    if m:
        cnum  = int(m.group(1))
        fidx  = int(m.group(2))
        csv_c = cnum - 1
        rep_frames[csv_c] = {'frame': fidx, 'label': 'Cluster ' + str(cnum), 'fname': fname.name}
        print('  ' + fname.name + '  ->  XTC frame ' + str(fidx))

rep_traj = {}
for csv_c, info in rep_frames.items():
    fidx = info['frame']
    rep_traj[csv_c] = md.load_frame(str(TRAJECTORY), fidx, top=str(TOPOLOGY))
    print('  ' + info['label'] + ' (frame ' + str(fidx) + '): loaded')

print('Done -- 4 single frames loaded, no full trajectory in RAM')


In [ ]:
# CELL 18 - Binary contacts for each representative frame

def contacts_single_frame(frame_traj, hvglu3_idx, cesa3_idx,
                           cesa3_seq, cesa3_name, cesa3_chain,
                           hvglu3_seq, hvglu3_name,
                           cutoff_nm=0.45, chunk=50):
    n_hv = len(hvglu3_idx)
    n_ce = len(cesa3_idx)
    contact_matrix = np.zeros((n_hv, n_ce), dtype=bool)
    for start in range(0, n_hv, chunk):
        end      = min(start + chunk, n_hv)
        chunk_hv = hvglu3_idx[start:end]
        pairs    = np.array([(int(i), int(j)) for i in chunk_hv for j in cesa3_idx])
        dist, _  = md.compute_contacts(frame_traj, contacts=pairs,
                                        scheme='closest-heavy', periodic=True)
        contact_matrix[start:end, :] = (dist[0] < cutoff_nm).reshape(end - start, n_ce)
    rows = []
    for i, j in zip(*np.where(contact_matrix)):
        rows.append({'HVGlu3_resid':   int(hvglu3_seq[i]),
                     'HVGlu3_resname': hvglu3_name[i],
                     'CESA3_chain':    cesa3_chain[j],
                     'CESA3_resid':    int(cesa3_seq[j]),
                     'CESA3_resname':  cesa3_name[j]})
    return pd.DataFrame(rows), contact_matrix

rep_contacts = {}
rep_matrices = {}
for csv_c in sorted(rep_traj.keys()):
    label = rep_frames[csv_c]['label']
    print('Computing contacts for ' + label + '...')
    df, mat = contacts_single_frame(
        rep_traj[csv_c], hvglu3_idx, cesa3_idx,
        cesa3_seq, cesa3_name, cesa3_chain,
        hvglu3_seq, hvglu3_name, cutoff_nm=CUTOFF_NM, chunk=CHUNK)
    rep_contacts[csv_c] = df
    rep_matrices[csv_c] = mat
    out_csv = OUT_DIR / ('contacts_rep_' + label.replace(' ', '_') + '.csv')
    df.to_csv(out_csv, index=False)
    print('  ' + label + ': ' + str(len(df)) + ' contacts')
print('Done!')


In [ ]:
# CELL 19 - 4-panel side-by-side contact maps

chain_color = {'B': '#2166ac', 'C': '#d6604d', 'D': '#4dac26'}
n_B = (cesa3_chain == 'B').sum()
n_C = (cesa3_chain == 'C').sum()
n_D = (cesa3_chain == 'D').sum()
offset = {'B': 0, 'C': n_B, 'D': n_B + n_C}
seq_to_ypos = {}
for ch in ['B', 'C', 'D']:
    m = cesa3_chain == ch
    for li, sq in enumerate(cesa3_seq[m]):
        seq_to_ypos[(ch, int(sq))] = li + offset[ch]

fig, axes = plt.subplots(2, 2, figsize=(20, 14), sharey=True)
axes = axes.flatten()
pops = ['18.7%', '31.9%', '22.5%', '26.9%']

for ax_i, csv_c in enumerate(sorted(rep_contacts.keys())):
    ax = axes[ax_i]
    df = rep_contacts[csv_c].copy()
    df['y'] = df.apply(
        lambda r: seq_to_ypos.get((r.CESA3_chain, r.CESA3_resid), np.nan), axis=1)
    df = df.dropna(subset=['y'])
    for ch in ['B', 'C', 'D']:
        sub = df[df['CESA3_chain'] == ch]
        ax.scatter(sub['HVGlu3_resid'], sub['y'],
                   c=chain_color[ch], s=15, alpha=0.7, linewidths=0, label='Chain ' + ch)
    hl = df[df['HVGlu3_resid'].isin({74, 77})]
    ax.scatter(hl['HVGlu3_resid'], hl['y'], c='gold', s=90, zorder=5,
               marker='*', linewidths=0.4, edgecolors='black')
    for b in [n_B - 0.5, n_B + n_C - 0.5]:
        ax.axhline(b, color='black', lw=0.7, ls='--', alpha=0.4)
    label    = rep_frames[csv_c]['label']
    is_close = (csv_c == CLOSE_CLUSTER)
    bcolor, blw = ('navy', 2.5) if is_close else ('grey', 0.5)
    for spine in ax.spines.values():
        spine.set_edgecolor(bcolor)
        spine.set_linewidth(blw)
    suffix = '  <- CLOSE STATE' if is_close else ''
    ttl = label + ' (' + pops[csv_c] + ')' + suffix + '\n' + str(len(df)) + ' contacts at 4.5 A'
    ax.set_title(ttl, fontsize=11, weight='bold', color='navy' if is_close else 'black')
    ax.set_xlabel('HvGlu3 residue', fontsize=10)
    if ax_i % 2 == 0:
        ax.set_yticks([n_B/2, n_B + n_C/2, n_B + n_C + n_D/2])
        ax.set_yticklabels(['Chain B', 'Chain C', 'Chain D'], fontsize=9)
    if ax_i == 0:
        ax.legend(loc='upper right', fontsize=8, framealpha=0.6)

from matplotlib.lines import Line2D
star = Line2D([0],[0], marker='*', color='w', markerfacecolor='gold',
               markersize=12, markeredgecolor='black', label='LEU74 / VAL77 (CESA1 anchors)')
fig.legend(handles=[star], loc='lower center', fontsize=10, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('CESA3 - Representative Frame Contact Maps (4.5 A cutoff)\n'
             'Navy border = Cluster 3 close-binding state (T1)  |  star = CESA1 anchor residues',
             fontsize=13, weight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
out = OUT_DIR / 'contact_map_rep_4panel.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ' + out.name)


In [ ]:
# CELL 20 - Cross-cluster comparison: which HvGlu3 residues are unique to Cluster 3?

rep_hv = {}
for csv_c in sorted(rep_contacts.keys()):
    df = rep_contacts[csv_c]
    if len(df):
        rep_hv[csv_c] = df.groupby('HVGlu3_resid')['CESA3_resid'].count()
    else:
        rep_hv[csv_c] = pd.Series(dtype=int)

all_active = sorted(set().union(*[set(v.index) for v in rep_hv.values()]))
rows_cmp = []
for seq in all_active:
    pos  = np.where(hvglu3_seq == seq)[0]
    name = hvglu3_name[pos[0]] if len(pos) else '???'
    row  = {'HVGlu3_resid': seq, 'resname': name}
    for csv_c in sorted(rep_contacts.keys()):
        row['C' + str(csv_c+1) + '_n'] = int(rep_hv[csv_c].get(seq, 0))
    close_n  = row['C' + str(CLOSE_CLUSTER+1) + '_n']
    distal_n = max(row['C' + str(c+1) + '_n'] for c in range(4) if c != CLOSE_CLUSTER)
    row['close_only'] = (close_n > 0 and distal_n == 0)
    row['delta']      = close_n - distal_n
    rows_cmp.append(row)

close_col = 'C' + str(CLOSE_CLUSTER+1) + '_n'
df_cmp = pd.DataFrame(rows_cmp).sort_values(close_col, ascending=False)
df_cmp.to_csv(OUT_DIR / 'rep_frame_contact_comparison.csv', index=False)
print('Contact count per HvGlu3 residue per representative frame:')
print(df_cmp.to_string(index=False))
print()
print('Residues UNIQUE to Cluster 3 (close-binding) representative frame:')
cols = ['HVGlu3_resid', 'resname', close_col]
print(df_cmp[df_cmp['close_only']][cols].to_string(index=False))


In [ ]:
# CELL 21 - PyMOL selection strings for Cluster 3 interface

df_c3   = rep_contacts[CLOSE_CLUSTER]
hv_list = sorted(df_c3['HVGlu3_resid'].unique())
hv_sel  = '+'.join(str(r) for r in hv_list)
ce_sels = {}
for ch in ['B', 'C', 'D']:
    rlist = sorted(df_c3[df_c3['CESA3_chain'] == ch]['CESA3_resid'].unique())
    if rlist:
        ce_sels[ch] = '+'.join(str(r) for r in rlist)

print('=' * 65)
print('PyMOL commands - load cluster_3_frame_199.pdb first')
print('=' * 65)
print()
print('load cluster_3_frame_199.pdb, cesa3_close')
print()
print('# HvGlu3 interface (chain A)')
print('select hvglu3_if, chain A and resi ' + hv_sel)
print('show sticks, hvglu3_if')
print('color cyan, hvglu3_if')
print()
for ch, sel in ce_sels.items():
    print('select cesa3_' + ch + '_if, chain ' + ch + ' and resi ' + sel)
    print('show sticks, cesa3_' + ch + '_if')
print()
print('color blue,   chain A   # HvGlu3')
print('color marine, chain B   # CESA3 subunit B')
print('color salmon, chain C   # CESA3 subunit C')
print('color green,  chain D   # CESA3 subunit D')
print()
print('select anchor, chain A and resi 74+77')
print('show sticks, anchor')
print('color yellow, anchor')
print()
print('show surface, chain B or chain C or chain D')
print('set transparency, 0.4')
print()
n_hv_if = len(hv_list)
print('HvGlu3 interface residues: ' + str(n_hv_if))
for ch, sel in ce_sels.items():
    print('CESA3 chain ' + ch + ' interface: ' + str(len(sel.split('+'))) + ' residues')


In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3/ContactMap')

for c in range(1, 5):
    df = pd.read_csv(OUT_DIR / f'contacts_cluster{c}_freq30.csv')
    print(f"Cluster {c} chain breakdown (frequency >= 0.5):")
    sub = df[df['frequency'] >= 0.5]
    print(sub['CESA3_chain'].value_counts())
    print()

In [ ]:
import pandas as pd
from pathlib import Path

CESA1_BASE = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA1/result_CESA1_ClusPro/CESA1_prod/Backbone_xtc')
df_cesa1_summary = pd.read_csv(CESA1_BASE / 'hvglu3_cesa_distance_CESA1_summary.csv')
print(df_cesa1_summary[['trajectory', 'mean_A', 'min_A', 'last20pct_mean_A']].round(2).to_string(index=False))
print(f"\nOverall minimum last-20% mean across CESA1 replicas: {df_cesa1_summary['last20pct_mean_A'].min():.2f} Å")